# Module 5: Agents & Tool Use Walkthrough

Build a working ReAct agent step by step — from a single tool call to a full multi-turn loop.


In [ ]:
import sys, os, json, math
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from anthropic import Anthropic
from dotenv import load_dotenv
load_dotenv()

client = Anthropic()
MODEL  = 'claude-haiku-4-5-20251001'
print('Client ready.')

## 1. Define a Tool

Tools are JSON Schema descriptions. Claude reads the schema and decides when to call the tool.

In [ ]:
CALCULATOR_TOOL = {
    'name': 'calculator',
    'description': 'Evaluates a Python math expression. Supports: +,-,*,/,**,sqrt(),sin(),cos().',
    'input_schema': {
        'type': 'object',
        'properties': {
            'expression': {'type': 'string', 'description': 'e.g. sqrt(144) or 2**10'}
        },
        'required': ['expression']
    }
}

def calculator(expression: str) -> str:
    allowed = {k: v for k, v in math.__dict__.items() if not k.startswith('_')}
    try:
        return f'Result: {eval(expression, {"__builtins__": {}}, allowed)}'
    except Exception as e:
        return f'Error: {e}'

print('Tool defined:', CALCULATOR_TOOL['name'])

## 2. One-Shot Tool Call

Send a query with the tool available. If `stop_reason == 'tool_use'`, Claude wants to call it.

In [ ]:
query = 'What is the square root of 1764?'

response = client.messages.create(
    model=MODEL, max_tokens=512,
    tools=[CALCULATOR_TOOL],
    messages=[{'role': 'user', 'content': query}]
)

print('stop_reason:', response.stop_reason)
for block in response.content:
    print('block type:', block.type)
    if block.type == 'tool_use':
        print('  tool:', block.name)
        print('  input:', block.input)

## 3. Execute the Tool and Return the Result

Run the tool, then add the result back to `messages` as a `tool_result` block.

In [ ]:
messages = [{'role': 'user', 'content': query}]
messages.append({'role': 'assistant', 'content': response.content})

tool_results = []
for block in response.content:
    if block.type == 'tool_use':
        result = calculator(**block.input)
        print(f'Executed: {block.name}({block.input}) → {result}')
        tool_results.append({
            'type': 'tool_result',
            'tool_use_id': block.id,
            'content': result
        })

messages.append({'role': 'user', 'content': tool_results})

# Get final answer
final = client.messages.create(model=MODEL, max_tokens=256, tools=[CALCULATOR_TOOL], messages=messages)
print('\nFinal answer:', final.content[0].text)

## 4. Full ReAct Loop

The complete agent: loops until `stop_reason == 'end_turn'`.

In [ ]:
def run_agent(query: str, max_turns: int = 8) -> str:
    messages = [{'role': 'user', 'content': query}]
    print(f'Query: {query}\n')
    
    for turn in range(max_turns):
        response = client.messages.create(
            model=MODEL, max_tokens=512,
            tools=[CALCULATOR_TOOL], messages=messages
        )
        
        if response.stop_reason == 'end_turn':
            answer = next(b.text for b in response.content if hasattr(b, 'text'))
            print(f'Answer: {answer}')
            return answer
        
        if response.stop_reason == 'tool_use':
            messages.append({'role': 'assistant', 'content': response.content})
            tool_results = []
            for block in response.content:
                if block.type == 'tool_use':
                    result = calculator(**block.input)
                    print(f'  Tool: {block.name}({block.input}) → {result}')
                    tool_results.append({'type': 'tool_result', 'tool_use_id': block.id, 'content': result})
            messages.append({'role': 'user', 'content': tool_results})
    
    return 'Max turns reached.'

# Test it
run_agent('If a square has area 289, what is its perimeter?')

## 5. Your Turn

Add a second tool (e.g. `unit_convert`) and test a multi-tool query.

In [ ]:
# Add your tool here and test!
run_agent('What is 17 * 23, then add 100 to that result?')